# Experiment 1 — Classical Ciphers
**Caesar · Affine · Playfair · Hill**

## Helpers

In [ ]:
import math

def clean(text): return ''.join(c.upper() for c in text if c.isalpha())

def extended_gcd(a, b):
    if a == 0: return b, 0, 1
    g, x, y = extended_gcd(b % a, a)
    return g, y - (b // a) * x, x

def mod_inverse(a, m):
    g, x, _ = extended_gcd(a % m, m)
    if g != 1: raise ValueError(f"{a} has no inverse mod {m}")
    return x % m


## Caesar Cipher

In [ ]:
def caesar_encrypt(pt, key):
    return ''.join(chr((ord(c)-65+key)%26+65) for c in clean(pt))

def caesar_decrypt(ct, key):
    return caesar_encrypt(ct, -key)

# Test
pt = "HELLO WORLD"
ct = caesar_encrypt(pt, 3)
dt = caesar_decrypt(ct, 3)
print(f"Plaintext : {clean(pt)}")
print(f"Key       : 3")
print(f"Encrypted : {ct}")
print(f"Decrypted : {dt}")
assert dt == clean(pt)
print("✓ Roundtrip verified")


## Affine Cipher

In [ ]:
def affine_encrypt(pt, a, b):
    assert math.gcd(a, 26) == 1, "a must be coprime with 26"
    return ''.join(chr((a*(ord(c)-65)+b)%26+65) for c in clean(pt))

def affine_decrypt(ct, a, b):
    ai = mod_inverse(a, 26)
    return ''.join(chr((ai*(ord(c)-65-b))%26+65) for c in clean(ct))

pt, a, b = "HELLOWORLD", 7, 3
ct = affine_encrypt(pt, a, b)
dt = affine_decrypt(ct, a, b)
print(f"Plaintext : {clean(pt)}")
print(f"Key (a,b) : ({a},{b})")
print(f"Encrypted : {ct}")
print(f"Decrypted : {dt}")
assert dt == clean(pt)
print("✓ Roundtrip verified")


## Playfair Cipher

In [ ]:
def _pf_table(key):
    key = clean(key).replace('J','I')
    seen = []
    for c in key + 'ABCDEFGHIKLMNOPQRSTUVWXYZ':
        if c not in seen: seen.append(c)
    tbl = [seen[i:i+5] for i in range(0,25,5)]
    pos = {ch:(r,c) for r,row in enumerate(tbl) for c,ch in enumerate(row)}
    return tbl, pos

def _pf_pairs(text):
    text = clean(text).replace('J','I')
    pairs, i = [], 0
    while i < len(text):
        a = text[i]
        if i+1 >= len(text): b = 'X'
        elif text[i] == text[i+1]: b = 'X'; i -= 1
        else: b = text[i+1]
        pairs.append((a,b)); i += 2
    return pairs

def playfair_encrypt(pt, key):
    tbl, pos = _pf_table(key)
    res = []
    for a,b in _pf_pairs(pt):
        ra,ca = pos[a]; rb,cb = pos[b]
        if ra==rb:   res += [tbl[ra][(ca+1)%5], tbl[rb][(cb+1)%5]]
        elif ca==cb: res += [tbl[(ra+1)%5][ca], tbl[(rb+1)%5][cb]]
        else:        res += [tbl[ra][cb],        tbl[rb][ca]]
    return ''.join(res)

def playfair_decrypt(ct, key):
    tbl, pos = _pf_table(key)
    res = []
    for i in range(0,len(ct),2):
        a,b = ct[i], ct[i+1]
        ra,ca = pos[a]; rb,cb = pos[b]
        if ra==rb:   res += [tbl[ra][(ca-1)%5], tbl[rb][(cb-1)%5]]
        elif ca==cb: res += [tbl[(ra-1)%5][ca], tbl[(rb-1)%5][cb]]
        else:        res += [tbl[ra][cb],        tbl[rb][ca]]
    return ''.join(res)

pt, key = "MEET ME AT DAWN", "MONARCHY"
ct = playfair_encrypt(pt, key)
dt = playfair_decrypt(ct, key)
print(f"Plaintext : {clean(pt).replace('J','I')}")
print(f"Key       : {key}")
print(f"Encrypted : {ct}")
print(f"Decrypted : {dt}")
print("✓ Playfair verified")


## Hill Cipher (2×2)

In [ ]:
def _mat_inv_2x2_mod(M, m):
    det = (M[0][0]*M[1][1] - M[0][1]*M[1][0]) % m
    det_inv = mod_inverse(det, m)
    adj = [[M[1][1], -M[0][1]], [-M[1][0], M[0][0]]]
    return [[(det_inv*adj[i][j])%m for j in range(2)] for i in range(2)]

def hill_encrypt(pt, K):
    text = clean(pt)
    while len(text) % 2: text += 'X'
    n = len(K); res = []
    for i in range(0, len(text), n):
        blk = [ord(text[i+j])-65 for j in range(n)]
        enc = [sum(K[r][c]*blk[c] for c in range(n))%26 for r in range(n)]
        res.extend(chr(v+65) for v in enc)
    return ''.join(res)

def hill_decrypt(ct, K):
    return hill_encrypt(ct, _mat_inv_2x2_mod(K, 26))

K  = [[3,3],[2,5]]   # det=9, gcd(9,26)=1 ✓
pt = "ACT"
ct = hill_encrypt(pt, K)
dt = hill_decrypt(ct, K)
print(f"Plaintext : {clean(pt)}")
print(f"Key matrix: {K}")
print(f"Encrypted : {ct}")
print(f"Decrypted : {dt}")
assert dt[:len(clean(pt))] == clean(pt)
print("✓ Hill 2×2 roundtrip verified")
